# LC 300 — Longest Increasing Subsequence
**Day 38 | 1D Dynamic Programming | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> For each element, ask: "what is
the longest increasing subsequence ending exactly here?"  That
is <code>1 + max(dp[j])</code> for all earlier elements smaller
than the current one. The O(n log n) trick maintains a
<em>tails</em> array where <code>tails[i]</code> is the smallest
tail of any increasing subsequence of length <code>i+1</code>.
</div>

## Official Problem Statement

Given an integer array `nums`, return the length of the longest
strictly increasing subsequence.

**Constraints:**
- `1 <= nums.length <= 2500`
- `-10^4 <= nums[i] <= 10^4`

**Follow-up:** Can you come up with an O(n log n) solution?

## What This Is Actually Asking

Find the longest chain of numbers in the array where each number
is strictly greater than the one before it in the chain. The
elements do not need to be contiguous — we may skip elements.
The O(n²) DP approach checks every pair; the O(n log n) approach
maintains a sorted `tails` list and uses binary search to
efficiently find where each number fits. The length of `tails`
at the end is the LIS length.

## Walk Through an Example by Hand

**Input:** `nums = [10, 9, 2, 5, 3, 7, 101, 18]`

**O(n²) DP trace (dp[i] = LIS length ending at index i):**
```
i=0: num=10  dp[0]=1
i=1: num=9   no j<1 with nums[j]<9  => dp[1]=1
i=2: num=2   no smaller predecessors => dp[2]=1
i=3: num=5   j=2 (2<5) => dp[3]=dp[2]+1=2
i=4: num=3   j=2 (2<3) => dp[4]=dp[2]+1=2
i=5: num=7   j=2(2<7),j=3(5<7),j=4(3<7)
             => dp[5]=max(dp[2],dp[3],dp[4])+1=3
i=6: num=101 all < 101 => dp[6]=max(dp)+1=4
i=7: num=18  j=2,3,4,5 (2,5,3,7<18) => dp[7]=dp[5]+1=4
```
Result: `max(dp) = 4`  (e.g. 2→5→7→101)

**O(n log n) tails trace:**
```
num=10 tails=[10]
num=9  replace tails[0]=9   tails=[9]
num=2  replace tails[0]=2   tails=[2]
num=5  append              tails=[2,5]
num=3  replace tails[1]=3  tails=[2,3]
num=7  append              tails=[2,3,7]
num=101 append             tails=[2,3,7,101]
num=18 replace tails[3]=18 tails=[2,3,7,18]
```
Result: `len(tails) = 4`

## The Picture

```
nums:  10   9   2   5   3   7  101  18
dp:     1   1   1   2   2   3    4   4
        |           |       |    |
        base        +--dp[2]+1   max=4

tails array (O(n log n)):
After each number:
  10  -> [10]
   9  -> [ 9]          (replaced 10)
   2  -> [ 2]          (replaced 9)
   5  -> [ 2, 5]       (appended)
   3  -> [ 2, 3]       (replaced 5)
   7  -> [ 2, 3, 7]    (appended)
 101  -> [ 2, 3, 7,101](appended)
  18  -> [ 2, 3, 7, 18](replaced 101)

len(tails) = 4  <-- LIS length
Note: tails is always sorted; bisect_left finds insert position.
```

## When To Use This Pattern

- When asked for the longest chain where each element beats the
  previous one, think LIS / DP.
- When n ≤ 2500, O(n²) DP is acceptable; when n > 2500,
  think O(n log n) with binary search.
- When maintaining a sorted structure and need the insertion
  point, think `bisect_left`.
- When the problem involves patience sorting or longest
  non-decreasing sequences, the tails trick generalises.
- When sub-problems ask "what is the best ending at position i?",
  think 1D DP with a nested comparison loop.

## The Approach

**O(n²):** Initialise `dp[i] = 1` for all i. For each index i,
scan every j < i: if `nums[j] < nums[i]`, update
`dp[i] = max(dp[i], dp[j] + 1)`. Return `max(dp)`.

**O(n log n):** Maintain a `tails` list. For each number, use
`bisect_left` to find the leftmost position in `tails` that is
>= current number. Replace that position (or append if past the
end). The length of `tails` is the answer.

In [ ]:
from typing import List
import bisect

In [ ]:
def test_harness(func):
    """Run test cases for lengthOfLIS."""
    cases = [
        # (nums, expected)
        ([10, 9, 2, 5, 3, 7, 101, 18], 4),  # classic
        ([0, 1, 0, 3, 2, 3],           4),  # 0,1,2,3
        ([7, 7, 7, 7, 7],              1),  # all equal (strict)
        ([1],                          1),  # single element
        ([1, 2, 3, 4, 5],              5),  # already sorted
        ([5, 4, 3, 2, 1],              1),  # fully decreasing
        ([3, 5, 6, 2, 5, 4, 19, 5, 6, 7, 12], 6),  # mixed
    ]
    passed = 0
    for nums, expected in cases:
        result = func(nums)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status}: nums={nums} "
                f"=> got {result}, expected {expected}"
            )
    total = len(cases)
    print(f"\nSummary: {passed}/{total} tests passed.")
    if passed == total:
        print("All tests PASSED!")

In [ ]:
def lengthOfLIS(nums: List[int]) -> int:
    """
    Return the length of the longest strictly increasing subsequence.

    Strategy (O(n log n)):
        Maintain sorted tails array.
        For each num, bisect_left to find position pos.
        If pos == len(tails): append num.
        Else: tails[pos] = num.
        Return len(tails).

    Args:
        nums: Input list of integers.

    Returns:
        Length of the longest strictly increasing subsequence.

    Examples:
        >>> lengthOfLIS([10,9,2,5,3,7,101,18])
        4
        >>> lengthOfLIS([7,7,7,7])
        1
    """
    # Debug: show input
    print(f"[DEBUG] nums = {nums}")

    # Debug: initialise tails
    print("[DEBUG] tails = []")

    # Debug: bisect_left usage
    print("[DEBUG] for num in nums:")
    print("[DEBUG]   pos = bisect.bisect_left(tails, num)")

    # Debug: append or replace
    print("[DEBUG]   if pos==len(tails): tails.append(num)")
    print("[DEBUG]   else: tails[pos] = num")

    # Debug: return
    print("[DEBUG] return len(tails)")

    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(lengthOfLIS)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (all subseqs) | O(2^n) | O(n) | Exponential, TLE |
| O(n²) DP | O(n²) | O(n) | Acceptable for n≤2500 |
| **O(n log n) tails** | **O(n log n)** | **O(n)** | **Optimal** |

## Real World Connection

At Citi, LIS appears in trade-sequence analysis: finding the
longest run of strictly increasing daily P&L to identify momentum
regimes. In AWS data pipelines, finding the longest increasing
version chain of dataset snapshots mirrors this problem exactly.
Data engineers use LIS-like logic in schema evolution tracking —
finding the longest sequence of backward-compatible schema
versions. The O(n log n) tails trick with `bisect` is a standard
Python pattern worth internalising because it appears in
scheduling, sequence alignment, and time-series analysis.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra